# 01 — Exploratory Data Analysis & Rheological Data Validation

## Chemical Engineering Background
Polymer solutions such as **Xanthan Gum (XG)** and **Carboxymethyl Cellulose (CMC)** exhibit pronounced non-Newtonian shear-thinning (pseudoplastic) flow behaviour. In biological, food, and petrochemical transport systems, understanding the relationship between shear rate ($\dot{\gamma}$), shear stress ($\tau$), and apparent viscosity ($\eta_a = \tau / \dot{\gamma}$) is essential for hydraulic pipe sizing, agitator design, and flow assurance.

### Objectives
1. Ingest rheological flow curve datasets (CSV/Excel format).
2. Enforce physical bounds and consistency validation ($\eta_a \approx \tau / \dot{\gamma}$).
3. Inspect temperature and concentration dependencies across CMC and XG.
4. Plot publication-grade flow curves on logarithmic scales.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR
from src.data_processing import (
    generate_synthetic_dataset, load_dataset, validate_dataset,
    data_summary, group_summary, filter_positive_shear_rate, add_log_columns
)
from src.visualization import plot_flow_curves, plot_flow_curves_by_fluid
print('Environment initialized successfully.')

## 1. Data Ingestion & Provenance
If experimental data is present in `data/raw/`, it will be loaded. Otherwise, a benchmark synthetic demonstration dataset is generated and explicitly labeled.

In [ ]:
raw_file = RAW_DATA_DIR / 'synthetic_rheology_data.csv'
if not raw_file.exists():
    df = generate_synthetic_dataset(save_path=raw_file, seed=42)
else:
    df = load_dataset(raw_file)

print(f'Loaded dataset with shape: {df.shape}')
df.head(10)

## 2. Rigorous Data Validation & Consistency Checks
Physical measurements must adhere to conservation and rheological laws:
- $\dot{\gamma} > 0$, $\tau > 0$, $\eta_a > 0$
- Internal consistency: $\left|\eta_a - \frac{\tau}{\dot{\gamma}}\right| / \eta_a < 5\%$

In [ ]:
report = validate_dataset(df)
print(report.summary())

## 3. Statistical Summaries by Experimental Group
Grouping by `(fluid_type, concentration_wt_pct, temperature_C)` reveals the span of shear rates and stresses.

In [ ]:
print('Overall Summary:')
display(data_summary(df))

print('\nCondition Groups:')
display(group_summary(df))

## 4. Flow Curves & Viscosity Curves Visualization
Plotting $\tau$ vs. $\dot{\gamma}$ and $\eta_a$ vs. $\dot{\gamma}$.

In [ ]:
fig1 = plot_flow_curves(df, save_name='notebook_flow_curves')
plt.show()

fig2 = plot_flow_curves_by_fluid(df, save_name='notebook_flow_curves_by_fluid')
plt.show()

### Physical Observations
1. **Shear Thinning**: All curves exhibit monotonic decreases in apparent viscosity with increasing shear rate (slope $< 1$ on $\log \tau$ vs $\log \dot{\gamma}$).
2. **Concentration Effect**: Higher polymer concentrations significantly increase the consistency index ($K$), shifting curves upward due to enhanced intermolecular chain entanglement.
3. **Temperature Effect**: Increasing temperature from $25^\circ\text{C}$ to $40^\circ\text{C}$ reduces viscosity across all shear rates, consistent with thermal agitation weakening non-covalent polymer network interactions.